### Transformer Machine Translation Lab
### Building a Transforner model for English → French translation.

### 1. Setup

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from collections import Counter
import math
import re

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

/Users/kaungkhantlin/Developer/2_2025/NLP/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


### 2. Load a dataset

In [3]:
raw = load_dataset("opus_books", "en-fr", split="train[:3000]")
raw = raw.train_test_split(test_size=0.2)
print(raw)

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 2400
    })
    test: Dataset({
        features: ['id', 'translation'],
        num_rows: 600
    })
})


### 3. Define a tokenization function

In [4]:
def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

### 4. Build vocabularies
##### Special tokens include
* &lt;pad&gt; - padding 
* &lt;unk&gt; - unknown
* &lt;sos&gt; - start of the sentence
* &lt;eos&gt; - end of the sentence

In [5]:
SPECIALS = ["<pad>", "<unk>", "<sos>", "<eos>"]

def build_vocab(texts, max_size=8000):
    counter = Counter()
    for t in texts:
        counter.update(tokenize(t))

    vocab = {tok: i for i, tok in enumerate(SPECIALS)}
    for word, _ in counter.most_common(max_size):
        if word not in vocab:
            vocab[word] = len(vocab)
    return vocab

source_vocab = build_vocab([ex["en"] for ex in raw["train"]["translation"]])

target_vocab = build_vocab([ex["fr"] for ex in raw["train"]["translation"]])

SRC_PAD = source_vocab["<pad>"]
TGT_PAD = target_vocab["<pad>"]

MAX_LEN = 40

### 5. Define an encoding function

In [6]:
def encode(text, vocab):
    tokens = tokenize(text)[:MAX_LEN-2]
    ids = [vocab["<sos>"]] + [vocab.get(t, vocab["<unk>"]) for t in tokens] + [vocab["<eos>"]]
    return ids + [vocab["<pad>"]] * (MAX_LEN - len(ids))

### 6. Define a Dataset class

In [7]:
class TranslationDataset(Dataset):
    def __init__(self, data):
        self.pairs = data["translation"]

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src = encode(self.pairs[idx]["en"], source_vocab)
        tgt = encode(self.pairs[idx]["fr"], target_vocab)
        return torch.tensor(src), torch.tensor(tgt)

train_ds = TranslationDataset(raw["train"])
val_ds = TranslationDataset(raw["test"])

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

### 7. Positional encoding

In [8]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

### 8. Transformer model

In [9]:
class TransformerMT(nn.Module):
    def __init__(self, source_vocab_size, target_vocab_size,
        d_model=128, nhead=4, num_layers=2):
        super().__init__()

        self.src_embed = nn.Embedding(source_vocab_size, d_model, padding_idx=SRC_PAD)
        self.tgt_embed = nn.Embedding(target_vocab_size, d_model, padding_idx=TGT_PAD)
        self.pos_enc = PositionalEncoding(d_model)

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dim_feedforward=256,
            batch_first=True
        )

        self.fc = nn.Linear(d_model, target_vocab_size)

    def forward(self, src, tgt):
        src_emb = self.pos_enc(self.src_embed(src))
        tgt_emb = self.pos_enc(self.tgt_embed(tgt))
        out = self.transformer(src_emb, tgt_emb)
        return self.fc(out)

model = TransformerMT(len(source_vocab), len(target_vocab)).to(DEVICE)

### Training setup

In [10]:
criterion = nn.CrossEntropyLoss(ignore_index=TGT_PAD)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

### Training loop

In [11]:
def train_epoch(model, loader):
    model.train()
    total_loss = 0

    for src, tgt in loader:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)

        # Shift target for teacher forcing
        decoder_input = tgt[:, :-1]
        decoder_target = tgt[:, 1:]

        optimizer.zero_grad()
        out = model(src, decoder_input)

        # Safety check for teaching/debugging
        assert out.shape[1] == decoder_target.shape[1], \
            f"Length mismatch: {out.shape} vs {decoder_target.shape}"

        loss = criterion(
            out.contiguous().view(-1, out.size(-1)),
            decoder_target.contiguous().view(-1)
        )
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

### Train

In [12]:
for epoch in range(5):
    loss = train_epoch(model, train_loader)
    print(f"Epoch {epoch+1}: Loss={loss:.4f}")

Epoch 1: Loss=7.1617
Epoch 2: Loss=6.3104
Epoch 3: Loss=5.8096
Epoch 4: Loss=5.2595
Epoch 5: Loss=4.7742


### Greedy decoding

In [13]:
inv_target_vocab = {i: w for w, i in target_vocab.items()}

def translate(sentence):
    model.eval()
    src = torch.tensor([encode(sentence, source_vocab)]).to(DEVICE)
    tgt_ids = [target_vocab["<sos>"]]

    for _ in range(MAX_LEN):
        target_tensor = torch.tensor([tgt_ids]).to(DEVICE)
        out = model(src, target_tensor)
        next_id = out[0, -1].argmax().item()
        if next_id == target_vocab["<eos>"]:
           break
        tgt_ids.append(next_id)

    return " ".join(inv_target_vocab[i] for i in tgt_ids[1:])


### Try Transformer out!!

In [14]:
print(translate("I like cats"))
print(translate("This class is very interesting"))

l l l l l l l l l l l l l l l l l l l l l l l l l l l l l l l l l l l l l l l l
l l l l l l l l l l l l l l l l l l l l l l l l l l l l l l l l l l l l l l l l
